In [3]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_anthropic.chat_models import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
#from google.colab import userdata
from langchain_core.tools import tool
from langgraph.graph import START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
import gradio as gr
import os
from PIL import Image
from collections import Counter
from typing import Annotated, TypedDict
import time, sys
from anthropic import Anthropic
from openai import OpenAI

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
sys.path.append('code')
from MolPropOp import *
#from lipinski_module import *
from docking_module import *

## adversary test

In [ ]:
tools = [grow_cycle, replace_groups, make_random_list, related, lipinski]

anthropic_key = os.getenv("ANTHROPIC_KEY")
client = Anthropic(api_key=anthropic_key)

openai_key = os.getenv("OPENAI_API_TOKEN")
model = ChatOpenAI(model_name="gpt-5.2", api_key=openai_key).bind_tools(tools)

def adversary(prompt: str):
    adversary_message = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=2000,
    system = '''
    You are a drug design assistant. You will recieve a proposal from  another model
    of novel molecules it has designed to bind to a particular protein target. The proposal will 
    include reasoning as to why the model thinks those molecules will bind well, and estimated 
    docking scores for each molecule. Your task is to analyze the proposal and find any flaws 
    in the reasoning or estimation of the docking scores. You should then suggest modifications 
    to the proposed molecules that would make them more likely to bind well, and provide reasoning 
    for why those modifications would help.

    The other model has access to the following tools, and you may suggest that it use these tools to 
    gather more information or test out modifications to the proposed molecules:

    - grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
                  a list of molecules and scores. 

    - replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, 
                      returning a list of new molecules and scores. 
    
    - make_random_list: this tool generates a list of substituents of specified length (num_items). 
    
    - related: this tool generates a list of molecules that are structurally related to a given molecule, 
                and may be useful for exploring the chemical space around promising molecules.

    - lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five.
    ''',
    messages=[
        {"role": "user", "content": prompt},])

    response = adversary_message.content[0].text

    return response

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends. *NOTE: if you choose to add a phenyl group to a molecule,
use the SMILES 'c7ccccc7', so that it does not interfere with other rings in the molecule that
may already use numbers 1-6 in their SMILES notation. 

The tools you have available include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

- related: this tool generates a list of molecules that are structurally related to a given molecule, and
           may be useful for exploring the chemical space around promising molecules you find in the input 
           data. It returns a list of related molecules and a few properties.

- lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five, 
            which is a set of guidelines for determining whether a molecule is likely to be an orally active 
            drug in humans. This tool can help you ensure that the molecules you are proposing not only have 
            good docking scores but also have properties that make them more likely to be successful as drugs.
            QED (quantitative estimate of drug-likeness) is a score between 0 and 1 that summarizes how 
            drug-like a molecule is, with 1 being the most drug-like. A higher QED score indicates that a 
            molecule has properties that are more consistent with known drugs, such as appropriate molecular 
            weight, lipophilicity, and number of hydrogen bond donors and acceptors.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## If you have identified good potential hits, evaluate the Lipinski properties of the proposed molecules 
and use that information to further refine your proposals, keeping in mind that you want to propose molecules 
that not only have good docking scores but also have good drug-like properties. 

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c['model']['messages'].content
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        #print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  #random_list, excluded_list = make_random_list(10)
  #first_list = sub_cycle(random_list, scoring_args)
  #context = ''
  #for smiles, score in first_list:
  #    context += f"{smiles}: {score}\n"

  with open('adversarial_set.md', 'r') as f:
    context = f.read()
  first_prompt = f'''
  Here is a list of molecules and their docking scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  #chat_history.append({'role': 'user', 'content': 'User sent protein name'})
  #chat_history.append({'role': 'assistant', 'content': 'Assistant running sub_cycle'})
  #chat_history.append({'role': 'user', 'content': first_prompt})
  #chat_history.append({'role': 'assistant', 'content': mes[-1]})

  return mes



In [ ]:
start_chat()

date_string = time.strftime("%Y-%m-%d_%H-%M-%S")
filename = f'../results/adversary_design_{date_string}.md'
with open(filename, 'w') as f:
    f.write(f'# Adversarial Design Session - {date_string}\n\n')

response_list = get_initial_prompt('HMGCR')
with open(filename, 'a') as f:
    f.write('# Initial model response:\n')
    text_av = response_list[-1][-1]+'\n'
    f.write(text_av)

text_av = ''
while text_av != 'Done':

    ant_response = adversary(response_list[-1][-1])
    with open(filename, 'a') as f:
        f.write('\n# Adversary feedback:\n')
        text_av = ant_response+'\n'
        f.write(text_av)

    _, _, response_list = chat_turn(ant_response)
    with open(filename, 'a') as f:
        f.write('\n# Model response:\n')
        text_av = response_list[-1][-1]+'\n'
        f.write(text_av)

Starting grow cycle with best score -8.6 for O=c1cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(F)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(Cl)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(Br)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C#N)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1([N+](=O)[O-])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(F)(F)(F))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(F)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(Cl)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(Br)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C#N)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2([N+](=O)[O-])ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(F)(F)(F))ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(F)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(Cl)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(Br)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(C#N)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2([N+](=O)[O-])cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2

[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Explicit valence for atom # 1 C, 6, is greater than permitted
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 6 7 8 9 10
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 7 8 9 10 11
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 12
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[15:51:03] Can't kekulize mol.  Unkekulized atoms: 2 3 13 14 15 16 21
[15:51:03] Can't kekulize mol.  Unkekulized 

O=c1c(F)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O=c1c(Cl)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -8.1
O=c1c(Br)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.5
O=c1c(C#N)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.0
O=c1c([N+](=O)[O-])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C(F)(F)(F))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1cc(-c2c(F)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(Cl)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(Br)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(C#N)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c([N+](=O)[O-])cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(C(F)(F)(F))cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2cc(F)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(Br)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(C#N)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2cc([N+](=O)[O-])ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2cc(C(F)(F)(F))ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2

[16:08:51] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:08:51] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:08:51] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:08:51] SMILES Parse Error: ring closure 1 duplicates bond between atom 1 and atom 2 for input: 'O=c1(c1ccccc1C(C(=O)[O-]))cc(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12'
[16:08:51] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 10 11 13 14 15
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 10 11 13 14 15
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 11 12 14 15 16
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 15 16 18 19 20
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 17 18 20 21 22
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 2 3 18 19 20 21 26
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 2 3 18 19 20 21 26
[16:08:51] Can't kekulize mol.  Unkekulized atoms: 2 3 19 20 21 22 27

got related molecules with smiles
2-(4-oxo-2-phenylchromen-5-yl)acetic acid
O=c1c(CC(C(=O)[O-]))c(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -7.7
O=c1c(CCC(=O)[O-])c(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -7.7
=========== New best score: -9.1 for O=c1cc(-c2cc(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12 ========
Best score: -9.1 for O=c1cc(-c2cc(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12
O=c1c(CCC(C(=O)[O-]))c(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(c1ccccc1C(C(=O)[O-]))c(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1c(CCc1ccccc1C(C(=O)[O-]))c(-c2cc(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: 0.0
O=c1cc(-c2c(CC(C(=O)[O-]))c(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(CCC(=O)[O-])c(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(CCC(C(=O)[O-]))c(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(c1ccccc1C(C(=O)[O-]))c(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: 0.0
O=c1cc(-c2c(CCc1ccccc1C(C(=O)[O-]))c(Cl)ccc2)oc2cccc(C(C(=O)[O-]))c12: 0.0
O=c1cc(-c2cc(Cl)c(CC(C(=O)[O-]))cc2)oc2cccc(C(C(

...

O=c1c(F)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.7
O=c1c(Cl)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(C)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(OC)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.6
O=c1c(C(C)(C)C)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(F)c(Cl)c(F)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(F)c(Cl)c(Cl)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(F)c(Cl)c(C)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2c(F)c(Cl)c(OC)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(F)c(Cl)c(C(C)(C)C)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(F)c(Cl)cc(F)c2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.2
O=c1cc(-c2c(F)c(Cl)cc(Cl)c2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(F)c(Cl)cc(C)c2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(F)c(Cl)cc(OC)c2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2c(F)c(Cl)cc(C(C)(C

[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Explicit valence for atom # 1 C, 6, is greater than permitted
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 6 8 10 11 12
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 6 8 10 11 12
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 6 8 10 11 12
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 6 8 10 11 12
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 7 9 11 12 13
[17:21:47] Can't kekulize mol.  Unkekulized atoms: 9 11 13 14 15
[17:21:47] Can't kekulize mol. 

O=c1c(F)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.7
O=c1c(F)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -8.2
O=c1c(Cl)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(Cl)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(C)c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(Br)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -9.0
O=c1c(C)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(C(F)(F)(F))c(-c2c(F)c(Cl)ccc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2c(F)c(Cl)c(F)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.1
O=c1c(CC)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2c(F)c(Cl)c(Cl)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -7.9
O=c1c(C(F)(F)(F))c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -7.4
O=c1cc(-c2c(F)c(Cl)c(C)cc2)oc2cccc(CCc1ccccc1C(C(=O)[O-]))c12: -8.5
O=c1c(C#N)c(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12: -8.0
O=

[18:29:09] SMILES Parse Error: extra open parentheses while parsing: O=c1cc(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccnc
[18:29:09] SMILES Parse Error: check for mistakes around position 35:
[18:29:09] c(Cl)ccc2)oc2ccc(C)c(CCc1ccnc
[18:29:09] ~~~~~~~~~~~~~~~~~~~~^
[18:29:09] SMILES Parse Error: Failed parsing SMILES 'O=c1cc(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccnc' for input: 'O=c1cc(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccnc'


O=c1cc(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccnc n1C(C(=O)[O-]))c12 bad
Best score: -9.5 for O=c1cc(-c2c(F)c(Cl)ccc2)oc2ccc(C)c(CCc1ccccc1C(C(=O)[O-]))c12
lipinski tool


## Ant first test

In [57]:
tools = [grow_cycle, replace_groups, make_random_list, related, lipinski]

anthropic_key = os.getenv("ANTHROPIC_KEY")
model = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=anthropic_key).bind_tools(tools)

openai_key = os.getenv("OPENAI_API_TOKEN")
client = OpenAI(api_key=openai_key)

def adversary(prompt: str):
    
    adversary_message = client.responses.create(
      model="gpt-5.2",
      instructions = '''
      You are a drug design assistant. You will recieve a proposal from  another model
      of novel molecules it has designed to bind to a particular protein target. The proposal will 
      include reasoning as to why the model thinks those molecules will bind well, and estimated 
      docking scores for each molecule. Your task is to analyze the proposal and find any flaws 
      in the reasoning or estimation of the docking scores. You should then suggest modifications 
      to the proposed molecules that would make them more likely to bind well, and provide reasoning 
      for why those modifications would help.

      The other model has access to the following tools, and you may suggest that it use these tools to 
      gather more information or test out modifications to the proposed molecules:

      - grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
                    a list of molecules and scores. 

      - replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, 
                        returning a list of new molecules and scores. 
      
      - make_random_list: this tool generates a list of substituents of specified length (num_items). 
      
      - related: this tool generates a list of molecules that are structurally related to a given molecule, 
                  and may be useful for exploring the chemical space around promising molecules.

      - lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five.
      ''',
      input=prompt
    )
    
    response = adversary_message.output_text

    return response

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends. *NOTE: if you choose to add a phenyl group to a molecule,
use the SMILES 'c7ccccc7', so that it does not interfere with other rings in the molecule that
may already use numbers 1-6 in their SMILES notation. 

The tools you have available include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

- related: this tool generates a list of molecules that are structurally related to a given molecule, and
           may be useful for exploring the chemical space around promising molecules you find in the input 
           data. It returns a list of related molecules and a few properties.

- lipinski: this tool evaluates a list of molecules for their drug-likeness based on Lipinski's rule of five, 
            which is a set of guidelines for determining whether a molecule is likely to be an orally active 
            drug in humans. This tool can help you ensure that the molecules you are proposing not only have 
            good docking scores but also have properties that make them more likely to be successful as drugs.
            QED (quantitative estimate of drug-likeness) is a score between 0 and 1 that summarizes how 
            drug-like a molecule is, with 1 being the most drug-like. A higher QED score indicates that a 
            molecule has properties that are more consistent with known drugs, such as appropriate molecular 
            weight, lipophilicity, and number of hydrogen bond donors and acceptors.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## If you have identified good potential hits, evaluate the Lipinski properties of the proposed molecules 
and use that information to further refine your proposals, keeping in mind that you want to propose molecules 
that not only have good docking scores but also have good drug-like properties. 

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c#['model']['messages'].content
      print(c)
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        #print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  #random_list, excluded_list = make_random_list(10)
  #first_list = sub_cycle(random_list, scoring_args)
  #context = ''
  #for smiles, score in first_list:
  #    context += f"{smiles}: {score}\n"

  with open('adversarial_set.md', 'r') as f:
    context = f.read()
  first_prompt = f'''
  Here is a list of molecules and their docking scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  #chat_history.append({'role': 'user', 'content': 'User sent protein name'})
  #chat_history.append({'role': 'assistant', 'content': 'Assistant running sub_cycle'})
  #chat_history.append({'role': 'user', 'content': first_prompt})
  #chat_history.append({'role': 'assistant', 'content': mes[-1]})

  return mes

In [1]:
messages[-1]

NameError: name 'messages' is not defined

In [54]:
messages.insert(1, chat_history[0][-2])

In [36]:
with open(filename, 'a') as f:
    f.write('\n #Model response:\n')
    f.write(response_list[0][-2]+'\n')

In [18]:
for i in range(len(response_list[-1])):
    with open('../results/adversary_design_2026-03-17_11-08-31.md', 'a') as f:
        f.write(f'\n\n## Model response sub-turn {i}:\n')
        if type(response_list[-1][i]) == str:
            out_text = response_list[-1][i]+ '\n'
            f.write(out_text)

In [ ]:
_, _, response_list = chat_turn(adv_response)
with open(filename, 'a') as f:
    f.write('\n# Model response:\n')
    text_av = response_list[-1][-1]+'\n'
    f.write(text_av)

{'model': {'messages': AIMessage(content=[{'text': 'This is an *excellent* and detailed critique. You\'ve identified several critical flaws in my reasoning, particularly around:\n\n1. **Anionic carboxylate scoring artifacts** — I conflated docking score with binding truth\n2. **Amide/HBD placement** — may be score bait rather than essential\n3. **Fluorine overconfidence** — likely driven by VDW/hydrophobic terms, not specific contacts\n4. **"Optimal" claims made without proper benchmarking** — no neutral acid, no structural analogs tested\n\nYou\'re absolutely right that I was overconfident given the docking-alone evidence. Let me systematically address your suggested experiments.\n\n## My plan:\n\nI\'ll test your highest-priority suggestions in this order:\n\n1. **Fix the acid-ionization artifact** (replace `[O-]` with neutral `O` and tetrazole)\n2. **Explore amide variants** (reverse amide, urea, sulfonamide)\n3. **Test the 3-position substituent** (H, F, Me, Et, OMe)\n4. **Probe "di

[17:03:44] SMILES Parse Error: syntax error while parsing: O=c1c(H)c(-c2c(F)c(F)c(NH(=O)H)cc2)oc2cccc(H(H(=O)[O-]))c12
[17:03:44] SMILES Parse Error: check for mistakes around position 7:
[17:03:44] O=c1c(H)c(-c2c(F)c(F)c(NH(=O)H)cc2)oc2ccc
[17:03:44] ~~~~~~^
[17:03:44] SMILES Parse Error: Failed parsing SMILES 'O=c1c(H)c(-c2c(F)c(F)c(NH(=O)H)cc2)oc2cccc(H(H(=O)[O-]))c12' for input: 'O=c1c(H)c(-c2c(F)c(F)c(NH(=O)H)cc2)oc2cccc(H(H(=O)[O-]))c12'
[17:03:44] Explicit valence for atom # 12 F, 4, is greater than permitted


In [20]:
filename = '../results/adversary_design_2026-03-17_11-08-31.md'
text_av = ''
while text_av != 'Done':

    adv_response = adversary(response_list[-1][-2])
    with open(filename, 'a') as f:
        f.write('\n# Adversary feedback:\n')
        text_av = adv_response+'\n'
        f.write(text_av)

    _, _, response_list = chat_turn(adv_response)
    with open(filename, 'a') as f:
        f.write('\n# Model response:\n')
        text_av = response_list[-1][-1]+'\n'
        f.write(text_av)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.1: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_015ygJ6NQGUzsxCruMb9oG4v. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CZ8yrxc8FNH5i8ojxdZB5'}

In [ ]:
start_chat()

date_string = time.strftime("%Y-%m-%d_%H-%M-%S")
filename = f'../results/adversary_design_{date_string}.md'
with open(filename, 'w') as f:
    f.write(f'# Adversarial Design Session - {date_string}\n\n')

response_list = get_initial_prompt('HMGCR')
with open(filename, 'a') as f:
    f.write('# Initial model response:\n')
    text_av = response_list[-1][-1]+'\n'
    f.write(text_av)

text_av = ''
while text_av != 'Done':

    adv_response = adversary(response_list[-1][-1])
    with open(filename, 'a') as f:
        f.write('\n# Adversary feedback:\n')
        text_av = adv_response+'\n'
        f.write(text_av)

    _, _, response_list = chat_turn(adv_response)
    with open(filename, 'a') as f:
        f.write('\n# Model response:\n')
        text_av = response_list[-1][-1]+'\n'
        f.write(text_av)

lipinski tool
Starting grow cycle with best score -8.6 for O=c1cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12.
O=c1(C(=O)[O-])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C#N)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1([N+](=O)[O-])cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(C(=O)[O-]))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(NC(=O)C)cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1(C(F)(F)(F))cc(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(=O)[O-])ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C#N)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2([N+](=O)[O-])ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(C(=O)[O-]))ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(NC(=O)C)ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2(C(F)(F)(F))ccccc2)oc2cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(C(=O)[O-])cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(C#N)cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2([N+](=O)[O-])cccc(C(C(=O)[O-]))c12 bad
O=c1cc(-c2ccccc2)oc2(C(C(=O)[O-]))cccc(C(C(=O)[O

[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 12
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 7 8 9 10 11
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 8 9 10 11 12
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 9 10 11 12 13
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 2 3 15 16 17 18 23
[11:08:41] Can't kekulize mol.  Unkekulized atoms: 2 3 14 15 16 17 22
[11:08:41] Can't kekulize mol.  Unke

O=c1c(C(=O)[O-])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C#N)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.0
O=c1c([N+](=O)[O-])c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(C(C(=O)[O-]))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1c(NC(=O)C)c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.5
O=c1c(C(F)(F)(F))c(-c2ccccc2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1cc(-c2c(C(=O)[O-])cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c(C#N)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2c([N+](=O)[O-])cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2c(C(C(=O)[O-]))cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2c(NC(=O)C)cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2c(C(F)(F)(F))cccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O=c1cc(-c2cc(C(=O)[O-])ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.1
O=c1cc(-c2cc(C#N)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.3
O=c1cc(-c2cc([N+](=O)[O-])ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1cc(-c2cc(C(C(=O)[O-]))ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.2
O=c1cc(-c2cc(NC(=O)C)ccc2)oc2cccc(C(C(=O)[O-]))c12: -8.5
O

[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Explicit valence for atom # 1 C, 6, is greater than permitted
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 8 9 10 15 16
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 7 8 9 14 15
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 8 9 10 15 16
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 9 10 11 16 17
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 10 11 12 17 18
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 9 10 11 16 17
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 2 3 19 20 21 22 27
[11:28:46] Can't kekulize mol.  Unkekulized atoms: 2 3 18 19 20 21 26
[11:28:46] Can't kekulize mol.  Unk

O=c1c(C(=O)[O-])c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1c(C#N)c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.6
O=c1c([N+](=O)[O-])c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1c(C(C(=O)[O-]))c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.4
O=c1c(NC(=O)CC)c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1c(C(F)(F)(F))c(-c2ccc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.0
O=c1cc(-c2c(C(=O)[O-])cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.7
O=c1cc(-c2c(C#N)cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1cc(-c2c([N+](=O)[O-])cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.9
O=c1cc(-c2c(C(C(=O)[O-]))cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.5
O=c1cc(-c2c(NC(=O)CC)cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1cc(-c2c(C(F)(F)(F))cc(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.6
O=c1cc(-c2cc(C(=O)[O-])c(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -8.2
O=c1cc(-c2cc(C#N)c(NC(=O)C)cc2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1cc(-c2cc([N+](=O)[O-])c(NC(=O)C)cc2)oc2cccc(C(C(=O

...

[16:00:31] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:00:31] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:00:31] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:00:31] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:00:31] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 9 11 13 18 19
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 8 10 12 17 18
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 2 4 20 21 22 23 28
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 2 4 20 21 22 23 28
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 2 4 22 23 24 25 30
[16:00:31] Can't kekulize mol.  Unkekulized atoms: 2 4 21 22 23 24 29
[16:00:31] Can't kekulize mol.  U

O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(O)c2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c([O-])c2)oc2cccc(C(C(=O)[O-]))c12: -7.8
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(N(C)C)c2)oc2cccc(C(C(=O)[O-]))c12: -8.0
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(NC)c2)oc2cccc(C(C(=O)[O-]))c12: -8.2
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(S)c2)oc2cccc(C(C(=O)[O-]))c12: -7.7
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(O)ccc(C(C(=O)[O-]))c12: -9.3
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c([O-])ccc(C(C(=O)[O-]))c12: -9.3
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(N(C)C)ccc(C(C(=O)[O-]))c12: -8.0
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(NC)ccc(C(C(=O)[O-]))c12: -8.2
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(S)ccc(C(C(=O)[O-]))c12: -7.7
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(O)cc(C(C(=O)[O-]))c12: -8.4
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc([O-])cc(C(C(=O)[O-]))c12: -8.4
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(N(C)C)cc(C(C(=O)[O-]))c12: -8.1
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(NC)cc(C(C(=O)[O-]))c12: -7.7
O=c1c(C)c(

[16:12:36] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:12:36] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:12:36] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:12:36] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:12:36] Explicit valence for atom # 1 C, 6, is greater than permitted
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 8 10 12 17 18
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 7 9 11 16 17
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 10 12 14 19 20
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 2 4 20 21 22 23 28
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 2 4 20 21 22 23 28
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 2 4 21 22 23 24 29
[16:12:36] Can't kekulize mol.  Unkekulized atoms: 2 4 20 21 22 23 28
[16:12:36] Can't kekulize mol.  

O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(F)c2)oc2cccc(C(C(=O)[O-]))c12: -7.9
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(Cl)c2)oc2cccc(C(C(=O)[O-]))c12: -9.1
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(CC)c2)oc2cccc(C(C(=O)[O-]))c12: -8.1
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(C)c2)oc2cccc(C(C(=O)[O-]))c12: -9.1
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)c(C(C)(C)C)c2)oc2cccc(C(C(=O)[O-]))c12: -8.4
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(F)ccc(C(C(=O)[O-]))c12: -9.2
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(Cl)ccc(C(C(=O)[O-]))c12: -8.1
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(CC)ccc(C(C(=O)[O-]))c12: -8.4
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(C)ccc(C(C(=O)[O-]))c12: -8.3
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2c(C(C)(C)C)ccc(C(C(=O)[O-]))c12: -8.2
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(F)cc(C(C(=O)[O-]))c12: -8.9
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(Cl)cc(C(C(=O)[O-]))c12: -8.0
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(CC)cc(C(C(=O)[O-]))c12: -7.7
O=c1c(C)c(-c2c(F)c(F)c(NC(=O)C)cc2)oc2cc(C)cc(C(C(=O)[O-]))c12: -8.0
O=c1c(C)c(-c2c

NameError: name 'ant_response' is not defined

## Lipinski test

In [ ]:
openai_key = os.getenv("OPENAI_API_TOKEN")

tools = [grow_cycle, replace_groups, make_random_list]

model = ChatOpenAI(model_name="gpt-5.2", api_key=openai_key).bind_tools(tools)

class State(TypedDict):
  messages: Annotated[list, add_messages]

def model_node(state: State) -> State:
  res = model.invoke(state['messages'])
  return {'messages': res}

builder = StateGraph(State)
builder.add_node('model', model_node)
builder.add_node('tools', ToolNode(tools))
builder.add_edge(START, 'model')
builder.add_conditional_edges('model', tools_condition)
builder.add_edge('tools',  'model')

graph = builder.compile()

sys_message = SystemMessage(content=f'''
{task_specific_prompt}

## You will first:
- Read the list of molecule SMILES and scores
- Ascertain any features of the molecules that contribute to the desired score. For example, if,
from one molecule to the next, the addition of an O group makes the score better.
- Gather all of these trends across all of the molecules.

## If you need additional information to ascertain the trends, such as more modified
molecules and their docking scores, you have tools you can call to generate new
molecules and get their docking scores. You can use these tools as many times as you want
to gather information on the trends.

the tools include:
                            
- grow_cycle: starts with a molecule SMILES and adds substituents to it, docks them, and returns 
              a list of molecules and scores. You can use this tool to further explore modifications
              to promising molecules that you find in the input data. You can provide a list of
              substituents to add, or use the predefined sets: e_withdraw (electron withdrawing),
              e_donate (electron donating), withdraw_with_linkers (electron withdrawing with linkers), 
              donate_with_linkers (electron donating with linkers). You can also generate a random list 
              of substituents with the make_random_list tool and use that as input to grow_cycle.

- replace_groups: starts with a molecule SMILES and replaces specific groups in it with new groups, returning a list of new
                  molecules and scores. This tool allows you to test specific hypotheses about how replacing certain
                  groups in a molecule might affect binding affinity. You can specify which groups to replace and
                  what to replace them with, or use the predefined sets of substituents mentioned above. You can also 
                  generate a random list of substituents with the make_random_list tool and use that as input to replace_groups.

- make_random_list: this tool generates a list of substituents of specified length (num_items). It draws from the predefined lists:
                    e_withdraw (electron withdrawing), e_donate (electron donating), withdraw_with_linkers 
                    (electron withdrawing with linkers), donate_with_linkers (electron donating with linkers). 
                    Use this tool when you want to get a broad sense of how different modifications affect binding affinity, 
                    without having a specific hypothesis in mind.

## Once you have ascertained the trends:
- Use the trends you learned to suggest 1-5 new molecules that obey the trends you found
and which should have a better score than the molecules in the list.
- Provide reasoning as to why you created those new molecules.
- Estimate the new scores.

## You may ask the user for clarification if needed, but try to use the tools to gather as much information as you
can before asking for clarification.

## In further turns, you will also receive feedback from an adversary model that is trying to find flaws 
in your reasoning and suggest improvements to your proposed molecules. You should use this feedback to 
refine your understanding of the trends, run new experiments with the tools to gather more information, 
and improve your proposed molecules in subsequent turns.

## Once you have reached a point where you think you have proposed the best possible molecules based on 
the trends, tool results and the adversary feedback, reply with only one word: "Done". This will signal that you have 
finished the task and will not propose any more molecules.
''')

global messages
messages = [sys_message]

#@spaces.GPU
def start_chat():
  '''
  '''
  global chat_history, messages, reasoning
  chat_history = []
  reasoning = []
  messages = [sys_message]

#@spaces.GPU
def chat_turn(prompt: str):
  '''
  '''
  global chat_history, messages, reasoning
  human_message = HumanMessage(content=prompt)
  messages.append(human_message)
  local_history = [prompt]

  input = {
      'messages' : messages
  }

  for c in graph.stream(input):
    try:
      ai_mes = c['model']['messages'].content
      messages.append(AIMessage(ai_mes))
      if ai_mes != '':
        print(f'message is {ai_mes}')
        local_history.append(ai_mes)
    except:
      pass
    try:
      if os.path.exists('current_image.png'):
        if os.path.getmtime('current_image.png') > time.time() - 30:
          img = Image.open('current_image.png')
        else:
          img = None
      else:
        img = None
    except:
      img = None
    try:
      reasoning.append(c['tools']['messages'][0].content)
    except:
      pass

  if len(local_history) != 2:
    local_history.append('no message')

  #chat_history.append({'role': 'user', 'content': local_history[0]})
  #chat_history.append({'role': 'assistant', 'content': local_history[1]})
  chat_history.append(local_history)
  return '', img, chat_history

def get_initial_prompt(protein):
  '''
  '''
  scoring_args[1] = protein
  random_list, excluded_list = make_random_list(10)
  first_list = sub_cycle(random_list, scoring_args)
  context = ''
  for smiles, score in first_list:
      context += f"{smiles}: {score}\n"

  first_prompt = f'''
  Here is a list of molecules and their scores:
  {context}\n'''

  blank, img, mes = chat_turn(first_prompt)
  #print(mes[-1])
  
  return mes

dudes = ['IGF1R', 'JAK2', 'KIT', 'LCK', 'MAPK14', 'MAPKAPK2', 'MET', 'PTK2', 'PTPN1', 'SRC', 'ABL1', 'AKT1', 'AKT2', 'CDK2', 'CSF1R', 'EGFR', 'KDR', 'MAPK1', 'FGFR1', 'ROCK1', 'MAP2K1', 'PLK1',
         'HSD11B1', 'PARP1', 'PDE5A', 'PTGS2', 'ACHE', 'MAOB', 'CA2', 'GBA', 'HMGCR', 'NOS1', 'REN', 'DHFR', 'ESR1', 'ESR2', 'NR3C1', 'PGR', 'PPARA', 'PPARD', 'PPARG',
        'AR','THRB','ADAM17', 'F10', 'F2', 'BACE1', 'CASP3', 'MMP13', 'DPP4', 'ADRB1', 'ADRB2', 'DRD2', 'DRD3','ADORA2A','CYP2C9', 'CYP3A4', 'HSP90AA1']


In [19]:
import importlib
importlib.reload(sys.modules['lipinski_module'])
from lipinski_module import *
importlib.reload(sys.modules['MolPropOp'])
from MolPropOp import *

In [24]:
start_chat()

In [25]:
get_initial_prompt('alogp')

Starting sub cycle on base rings for protein alogp.
c1(CC=C(I))ccccc1: 3.1778000000000013
c1(N(F))ccccc1: 1.9829999999999999
c1(C(=O)O(C#N))ccccc1: 1.3244799999999999
c1(CC([Si](C)(C)C))ccccc1: 3.567300000000002
c1(Br)ccccc1: 2.4491000000000005
c1(S(c5ccccc5))ccccc1: 3.8378000000000014
c1(CC=C(O))ccccc1: 2.3008000000000006
c1(C=CC(C(Cl)(Cl)(Cl)))ccccc1: 4.460100000000002
c1(C=C(C(C)C))ccccc1: 3.355800000000002
c1(C(=O)(OC(=O)C))ccccc1: 1.3899000000000001
n1c(CC=C(I))cccc1: 2.572800000000001
n1c(N(F))cccc1: 1.378
n1c(C(=O)O(C#N))cccc1: 0.7194799999999999
n1c(CC([Si](C)(C)C))cccc1: 2.9623000000000017
n1c(Br)cccc1: 1.8440999999999999
n1c(S(c5ccccc5))cccc1: 3.232800000000001
n1c(CC=C(O))cccc1: 1.6958
n1c(C=CC(C(Cl)(Cl)(Cl)))cccc1: 3.855100000000002
n1c(C=C(C(C)C))cccc1: 2.750800000000001
n1c(C(=O)(OC(=O)C))cccc1: 0.7848999999999999
n1cc(CC=C(I))ccc1: 2.572800000000001
n1cc(N(F))ccc1: 1.378
n1cc(C(=O)O(C#N))ccc1: 0.7194799999999999
n1cc(CC([Si](C)(C)C))ccc1: 2.9623000000000017
n1cc(Br)ccc1:

[['\n  Here is a list of molecules and their scores:\n  c1(CC=C(I))ccccc1: 3.1778000000000013\nc1(N(F))ccccc1: 1.9829999999999999\nc1(C(=O)O(C#N))ccccc1: 1.3244799999999999\nc1(CC([Si](C)(C)C))ccccc1: 3.567300000000002\nc1(Br)ccccc1: 2.4491000000000005\nc1(S(c5ccccc5))ccccc1: 3.8378000000000014\nc1(CC=C(O))ccccc1: 2.3008000000000006\nc1(C=CC(C(Cl)(Cl)(Cl)))ccccc1: 4.460100000000002\nc1(C=C(C(C)C))ccccc1: 3.355800000000002\nc1(C(=O)(OC(=O)C))ccccc1: 1.3899000000000001\nn1c(CC=C(I))cccc1: 2.572800000000001\nn1c(N(F))cccc1: 1.378\nn1c(C(=O)O(C#N))cccc1: 0.7194799999999999\nn1c(CC([Si](C)(C)C))cccc1: 2.9623000000000017\nn1c(Br)cccc1: 1.8440999999999999\nn1c(S(c5ccccc5))cccc1: 3.232800000000001\nn1c(CC=C(O))cccc1: 1.6958\nn1c(C=CC(C(Cl)(Cl)(Cl)))cccc1: 3.855100000000002\nn1c(C=C(C(C)C))cccc1: 2.750800000000001\nn1c(C(=O)(OC(=O)C))cccc1: 0.7848999999999999\nn1cc(CC=C(I))ccc1: 2.572800000000001\nn1cc(N(F))ccc1: 1.378\nn1cc(C(=O)O(C#N))ccc1: 0.7194799999999999\nn1cc(CC([Si](C)(C)C))ccc1: 2.962